In [6]:
# dowbloading the dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-06-14 09:53:00--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.1’

input.txt.1         100%[===================>]   1.06M  --.-KB/s    in 0.008s  

2026-06-14 09:53:00 (125 MB/s) - ‘input.txt.1’ saved [1115394/1115394]



In [7]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(66)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
# hyper-parameters
batch_size = 64
block_size = 256
n_emb = 384
n_head = 6
dropout_rate = 0.2
n_layer = 6
eval_interval = 500
eval_iters = 66
max_steps = 5000
learning_rate = 6e-4

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# extracting unique characters
print("chars in dataset: ", len(text))
chars = list(set(text))
voc_size = len(chars)
print(voc_size, chars)

# creating mapping
ctoi = {ch:i for i,ch in enumerate(chars)}
itoc = {i:ch for i,ch in enumerate(chars)}

# encode / decode strings
encode = lambda s: [ctoi[c] for c in s]
decode = lambda l: ''.join([itoc[i] for i in l])

# using torch.tensor
data = torch.tensor(encode(text), dtype=torch.long)

# spliting into train and test data
n = int(0.9 * len(data))
train_data = data[:n]
test_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else test_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i : i + block_size] for i in ix])
    y = torch.stack([data[i + 1 : i + block_size + 1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'test']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

# one head self attention
class head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_emb, head_size, bias=False)
        self.query = nn.Linear(n_emb, head_size, bias=False)
        self.value = nn.Linear(n_emb, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x) # (B,T,head_size)
        q = self.query(x) # (B,T,head_size)
        v = self.value(x) # (B,T,head_size)

        wei = q @ k.transpose(-2, -1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        out = wei @ v # (B,T,head_size)
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, n_emb)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedForward(nn.Module):
    def __init__(self, n_emb):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_emb, 4 * n_emb),
            nn.ReLU(),
            nn.Linear(4 * n_emb, n_emb),
            nn.Dropout(dropout_rate)
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_emb, n_head):
        super().__init__()
        head_size = n_emb // n_head
        self.sa_head = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_emb)
        self.ln1 = nn.LayerNorm(n_emb)
        self.ln2 = nn.LayerNorm(n_emb)

    def forward(self, x):
        x = x + self.sa_head(self.ln1(x))
        x = x +self.ffwd(self.ln2(x))
        return x

class BigramLangModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(voc_size, n_emb)
        self.position_embedding_table = nn.Embedding(block_size, n_emb)
        self.blocks = nn.Sequential(*[Block(n_emb, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_emb)
        self.lm_head = nn.Linear(n_emb, voc_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        final_emb = tok_emb + pos_emb
        final_emb = self.blocks(final_emb)
        final_emb = self.ln_f(final_emb)
        logits = self.lm_head(final_emb)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_trunc = idx[:, -block_size:]
            # get predictions
            logits, loss = self(idx_trunc)
            # we need the embedding of the last step only
            logits = logits[:, -1, :] # (B, C)
            # we convert the predictions into probabilties
            probs = F.softmax(logits, dim=-1) # (B, C)
            # we roll a dies based on probabities to get next token
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # we add the token we got to idx
            idx = torch.cat((idx, idx_next), dim=1) # (B, T + 1)
        return idx

model = BigramLangModel()
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
print("model initialized and loaded, roger that!")

chars in dataset:  1115394
65 ['j', ',', 'E', 'e', 'U', 'f', ':', 'd', 'b', 'k', 'q', 'r', 'w', 'v', '?', 'x', 'h', 'm', '.', "'", 'I', 'Z', 'F', 'N', 'C', 'a', 'i', 's', 'g', '-', 'M', '$', 'z', 'P', 'J', 'D', 'H', 'K', '\n', 'B', 'n', 'R', '!', 'W', 'c', 'Q', 't', 'A', 'V', ' ', ';', 'X', 'o', 'S', 'O', 'L', 'T', 'G', '3', 'l', 'Y', 'y', 'u', '&', 'p']
model initialized and loaded, roger that!


In [8]:
for step in range(max_steps):
    if step % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {step}: train loss {losses['train']:.4f}, test loss {losses['test']:.4f}")

    xb, yb = get_batch('train')

    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

torch.save(model.state_dict(), 'shakespeare_from_temu.pth')
print("cargo Secured, neural state extracted and locked as 'shakespeare_from_temu.pth'!")

step 0: train loss 4.4070, test loss 4.4051
step 500: train loss 1.8236, test loss 1.9598
step 1000: train loss 1.4429, test loss 1.6399
step 1500: train loss 1.3071, test loss 1.5524
step 2000: train loss 1.2211, test loss 1.5096
step 2500: train loss 1.1530, test loss 1.4833
step 3000: train loss 1.1016, test loss 1.4845
step 3500: train loss 1.0531, test loss 1.4917
step 4000: train loss 0.9953, test loss 1.5028
step 4500: train loss 0.9443, test loss 1.5309
cargo Secured, neural state extracted and locked as 'shakespeare_from_temu.pth'!


In [9]:
print("shakepeare starts to yap...\n")
context = torch.zeros((1, 1), dtype=torch.long).to(device)
print(decode(model.generate(idx = context, max_new_tokens = 6000)[0].tolist()))

shakepeare starts to yap...

jply.
And to prison, send them for the such feet
At hairly ambracging mousnet. They said
Forbear this shame,
Did it e'er no others in his ground.

ESCALUNT:
What were trebled,
Upon your talk?

ELBOW:
I am always done; I say no soulder say.
That you first may have safety with
sil; end his foul sentence.

JOHN OFF:
Lord, I am sure, sir, I pray it in me;
Here care you not slay the cat your fire.

ESCALUS:
Helpose! Where is the most prophecy?

ELBOW:
Ay, if the young prisoners weakness may losely kiss the
tresperous.

ESCALUS:
I am a little here upon you.

ESCALUS:
Why, sir, with all her appearance. Be ready sister
those far tops here, I crave the fire even to pursurp
on: 'twixt the crutch. Pity hurs it,'For such is a power, stand
to 's thungry brown. This chossal, I cry
your city is a father gap on then.
We do revolt any name to safegnage him,
bringing the into the water upon him.

First Citizen:
Thy masters.

Second Conspirator:
Hath corns! why?

Second Capul